# Week 6 Homework: Your Answers vs. Your Vibe-Coded Dashboard

**Week 4 dashboard link:** _(https://movie-dash.streamlit.app/)_

---

Two weeks ago you vibe-coded a MovieLens dashboard before you had the tools to judge it. Since then you've had the stats lecture (Week 5) and this week's EDA lecture. Now you do it the real way: answer the same 4 questions yourself, by hand, then audit the dashboard against your own work.

**Ground rules**

- 🚫 **No AI for code or answers in Parts 1 and 2.** Lecture notebooks, pandas/seaborn docs, Google, and Stack Overflow are all fair game. Write Part 3 in your own words too.
- 🔒 **Lock in before you peek.** Finish Parts 1 and 2 (answers written down) *before* you open your dashboard. If you look first, you'll anchor on its numbers and the comparison is worthless.
- 🔢 **Show your numbers.** Every answer needs actual values (counts, averages, titles), not just a vibe.
- ✏️ Written answers go in markdown cells (double-click to edit, Shift+Enter to render). Most are 2 to 4 sentences. Short and specific beats long and vague.

## The data

`movie_ratings.csv` has one row per rating: 100,000 ratings from 943 users on 1,682 movies.

- `user_id`, `movie_id`: who rated, which movie
- `rating`: 1 to 5 stars
- `timestamp`, `rating_year`: when the rating was given
- `age`, `gender`, `occupation`, `zip_code`: the rater
- `title`: movie title, with release year in parentheses
- `year`, `decade`: the movie's release year and decade
- `genres`: all of the movie's genres in one string, separated by the pipe character (a movie can have several)

## Setup

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Works whether this notebook sits in the week folder or inside exercise/
data_path = next(p for p in ["data/movie_ratings.csv", "../data/movie_ratings.csv"] if Path(p).exists()) # checks for the file in two possible locations
df = pd.read_csv(data_path)
df.head()

,user_id,movie_id,rating,timestamp,age,gender,occupation,zip_code,title,year,decade,genres,rating_year
0,196,242,3,1997-12-04 15:55:49,49,M,writer,55105,Kolya (1996),1996.0,1990.0,Comedy,1997
1,186,302,3,1998-04-04 19:22:22,39,F,executive,00000,L.A. Confidential (1997),1997.0,1990.0,Crime|Film-Noir|Mystery|Thriller,1998
2,22,377,1,1997-11-07 07:18:36,25,M,writer,40206,Heavyweights (1994),1994.0,1990.0,Children|Comedy,1997
3,244,51,2,1997-11-27 05:02:03,28,M,technician,80525,Legends of the Fall (1994),1994.0,1990.0,Drama|Romance|War|Western,1997
4,166,346,1,1998-02-02 05:33:16,47,M,educator,55113,Jackie Brown (1997),1997.0,1990.0,Crime|Drama,1998


---
# Part 1: Methodology

## 1.1 Know your data first

Before answering anything, check what you're working with: shape, column types, missing values, duplicate rows, and the range of the numeric columns.

In [2]:
# Your data checks here

# shape of the data 
print(df.shape)

# check missing values
df.isnull().sum()

(100000, 13)


user_id         0
movie_id        0
rating          0
timestamp       0
age             0
gender          0
occupation      0
zip_code        0
title           0
year           30
decade         30
genres          0
rating_year     0
dtype: int64

In [3]:
# check column types 
df.dtypes

user_id          int64
movie_id         int64
rating           int64
timestamp          str
age              int64
gender             str
occupation         str
zip_code           str
title              str
year           float64
decade         float64
genres             str
rating_year      int64
dtype: object

In [4]:
# check for dupilcated rows 
print("Duplicated rows:", df.duplicated().sum())

Duplicated rows: 0


In [5]:
# check for numeric column ranges
df.describe()

,user_id,movie_id,rating,age,year,decade,rating_year
count,100000.00000,100000.000000,100000.000000,100000.000000,99970.000000,99970.000000,100000.000000
mean,462.48475,425.530130,3.529860,32.969850,1987.748224,1982.780134,1997.471010
std,266.61442,330.798356,1.125674,11.562623,14.140014,13.820687,0.499161
min,1.00000,1.000000,1.000000,7.000000,1922.000000,1920.000000,1997.000000
25%,254.00000,175.000000,3.000000,24.000000,1986.000000,1980.000000,1997.000000
50%,447.00000,322.000000,4.000000,30.000000,1994.000000,1990.000000,1997.000000
75%,682.00000,631.000000,4.000000,40.000000,1996.000000,1990.000000,1998.000000
max,943.00000,1682.000000,5.000000,73.000000,1998.000000,1990.000000,1998.000000


In [6]:
# unique genres
df.genres.value_counts()

genres
Drama                        13257
Comedy                        9828
Comedy|Romance                5055
Drama|Romance                 4767
Action|Thriller               3550
                             ...  
Action|Adventure|Children        5
Action|Comedy|Crime|Drama        5
Crime|Film-Noir                  4
Fantasy                          1
Comedy|Documentary               1
Name: count, Length: 216, dtype: int64

In [7]:
# number of ratings based on year 
df.groupby('year').size()

year
1922.0       54
1926.0        2
1930.0       18
1931.0       44
1932.0       12
          ...  
1994.0     9034
1995.0    12389
1996.0    18744
1997.0    15384
1998.0      851
Length: 71, dtype: int64

**✏️ Data notes:** _(
  The numeric values appear reasonable overall. Ratings fill within the expected 1-5 range, and user ages range from 7 to 73. But we should still note there are 30 missing movie release year values, which also results in 30 missing decade values. Movie release years are concentrated in the more recent decades and rating_year only ranges from 1997 1998.)_

## 1.2 Decide before you chart

In Week 4 the questions were ambiguous on purpose. This is where *you* commit to an interpretation. Fill it in now, and update it if you change your mind while working (just say so).

**Q1 Genre breakdown**
- What you have to decide: What am I counting, ratings or movies? How do I handle movies that have several genres?
- My choice: count each unique movie once in every genre it belongs to.
- Why: Since we want to see the genre breakdown among movies, rating count would actually bias more popular movies. Having multiple genres preserves that information, but genre counts may add up to more than the total number of movies. 

**Q2 Genre satisfaction**
- What you have to decide: "Average rating" of what? How do I treat genres that have very little data?
- My choice: Each genre's mean should be calculated using all of its ratings, but when genre's are ranked, we should only consider ones with at least 100 ratings and 20 unique movies. 
- Why: we must consider our sample sizes when determining the mean ratings for genres. A genre such as unknown has 10 ratings across 2 movies, but the rating itself can seem misleading. But we should still show the genres that were filtered out.

**Q3 Ratings over time**
- What you have to decide: Is every year backed by the same amount of data? What do I do with movies that have no release year?
- My choice: Calculate the mean rating for each movie release year, exclude records with missing release years, and display the number of ratings behind each yearly average.
- Why: The years have very unequal sample sizes. Some years have only a few ratings, while the most represented year has 18,744. Showing counts makes the less reliable yearly averages visible. I will exclude the 30 ratings from 5 movies with missing release years because their years cannot be determined reliably, and imputing them could distort the trend.

**Q4 Best movies**
- What you have to decide: What is a movie's rating (the mean of what)? Is "at least 50" ≥ or >? What if there's a tie? 50 -> number of ratings, so how large should the sample size be?
- My choice: Calculate each movie’s mean user rating, keep movies with at least 50 ratings (>= 50), and rank them by mean rating. For ties, rank the movie with more ratings first.
- Why: Requiring 50 or more ratings prevents movies with only a few unusually high ratings from appearing as the best. Using rating count to break ties favors the result supported by more data.

---
# Part 2: Results (by hand, no AI)

Answer each question with a chart or table **and** a written answer with real numbers. Pick the chart type yourself and be ready to defend it in Part 3. Add as many code cells as you need.

In [8]:
%pip install plotly
%pip install plotly statsmodels nbformat --quiet
import plotly.express as px
import plotly.graph_objects as go


[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: /Users/ragibn/Library/CloudStorage/OneDrive-Personal/Documents/Programs/CUNY Tech Prep/Data Science - Fall/my-ds-fall-2026/.venv/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: /Users/ragibn/Library/CloudStorage/OneDrive-Personal/Documents/Programs/CUNY Tech Prep/Data Science - Fall/my-ds-fall-2026/.venv/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## Q1. Genre breakdown

**What's the distribution of genres among the movies that were rated?**

<details>
<summary>Stuck on movies with several genres? Hint</summary>

`genres` is one string like `Crime|Film-Noir|Mystery|Thriller`. To get one row per genre:

```python
genre_df = df.assign(genre=df['genres'].str.split('|')).explode('genre')
```

Heads up: after this, a movie with 3 genres takes up 3 rows. Think about what that does to anything you count or average. Exploding is fine for "how often does a genre show up", but be careful calling the result "market share" (the percentages add up to more than 100%).
</details>

In [9]:
# Q1: your code here

# Lets create a new df with only the movie id and the genres 
movie_genres = df[['movie_id', 'genres']].drop_duplicates()
movie_genres = movie_genres.assign(genres=movie_genres['genres'].str.split('|')).explode('genres')
movie_genres.head(10)


,movie_id,genres
0,242,Comedy
1,302,Crime
1,302,Film-Noir
1,302,Mystery
1,302,Thriller
2,377,Children
2,377,Comedy
3,51,Drama
3,51,Romance
3,51,War


In [10]:
# count unique movie_ids fo each genre, sort counts from largest to smallest
genre_count = movie_genres.groupby('genres')['movie_id'].nunique().reset_index(name='count').sort_values(by='count', ascending=False)
genre_count.head()

,genres,count
7,Drama,725
4,Comedy,505
0,Action,251
15,Thriller,251
13,Romance,247


**✏️ My answer (with numbers):** Drama is the most common genre, appearing in 725 of the 1,682 unique movies (43.1%), followed by Comedy with 505 movies (30.0%). Action and Thriller are tied next at 251 movies each (14.9%), followed by Romance with 247 movies (14.7%). I counted each unique movie once in every genre it belongs to, so these percentages overlap and do not add up to 100%.

## Q2. Genre satisfaction

**Which genres have the highest average rating? Which have the lowest?**

Show how much data is behind each average (for example, the count next to it). An average of 10 ratings and an average of 10,000 ratings are not the same kind of evidence.

In [11]:
# Q2: your code here

# lets create a genre_ratings df 
genre_ratings = (
  df.assign(genres=df['genres'].str.split('|'))
    .explode('genres')
)

genre_ratings[['movie_id', 'genres', 'rating']].head(15) 

,movie_id,genres,rating
0,242,Comedy,3
1,302,Crime,3
1,302,Film-Noir,3
1,302,Mystery,3
1,302,Thriller,3
2,377,Children,1
2,377,Comedy,1
3,51,Drama,2
3,51,Romance,2
3,51,War,2


In [12]:
# summarize ratings be genre 
genre_satisfaction = (
    genre_ratings.groupby('genres')
    .agg(
        average_rating=('rating', 'mean'), # apply mean function to the rating column and rename it to average_rating
        rating_count=('rating', 'count'), # apply count function to the rating column and rename it to rating_count
        movie_count=('movie_id', 'nunique') # apply nunique function to the movie_id column and rename it to movie_count
    )
    .reset_index() 
)

genre_satisfaction.head()

,genres,average_rating,rating_count,movie_count
0,Action,3.480245,25589,251
1,Adventure,3.503527,13753,135
2,Animation,3.576699,3605,42
3,Children,3.353244,7182,122
4,Comedy,3.394073,29832,505


In [13]:
# lets apply the genre rule: 
# keep genres with at least 100 ratings and 20 unique movies, then rank them by average rating.

qualified_genres = (
    genre_satisfaction[
        (genre_satisfaction['rating_count'] >= 100) &
        (genre_satisfaction['movie_count'] >= 20)
    ]
    .sort_values('average_rating', ascending=False) # sort the values by average_rating in descending order
    .reset_index(drop=True) # use drop=True to avoid adding the old index as a column
)

qualified_genres

,genres,average_rating,rating_count,movie_count
0,Film-Noir,3.921523,1733,24
1,War,3.815812,9398,71
2,Drama,3.687379,39895,725
3,Documentary,3.672823,758,50
4,Mystery,3.638132,5245,61
5,Crime,3.632278,8055,109
6,Romance,3.621705,19461,247
7,Western,3.613269,1854,27
8,Animation,3.576699,3605,42
9,Sci-Fi,3.560723,12730,101


In [14]:
# lets add a rank column to the qualified_genres df

qualified_genres['average_rating'] = qualified_genres['average_rating'].round(2) # two decimal places for average_rating
qualified_genres['rank'] = range(1, len(qualified_genres) + 1) 
 
qualified_genres = qualified_genres[
    ['rank', 'genres', 'average_rating', 'rating_count', 'movie_count'] # reorder the columns to have rank first, then genres, average_rating, rating_count, and movie_count
]

qualified_genres

,rank,genres,average_rating,rating_count,movie_count
0,1,Film-Noir,3.92,1733,24
1,2,War,3.82,9398,71
2,3,Drama,3.69,39895,725
3,4,Documentary,3.67,758,50
4,5,Mystery,3.64,5245,61
5,6,Crime,3.63,8055,109
6,7,Romance,3.62,19461,247
7,8,Western,3.61,1854,27
8,9,Animation,3.58,3605,42
9,10,Sci-Fi,3.56,12730,101


In [15]:
# lets create a table showing the excluded genres 
filtered_out_genres = (
    genre_satisfaction[
        (genre_satisfaction['rating_count'] < 100) |
        (genre_satisfaction['movie_count'] < 20)
    ]
    .sort_values('average_rating', ascending=False)
    .reset_index(drop=True)
)

filtered_out_genres['average_rating'] = (
    filtered_out_genres['average_rating'].round(2)
)

filtered_out_genres

,genres,average_rating,rating_count,movie_count
0,unknown,3.2,10,2


In [16]:
fig = px.bar(
    qualified_genres,
    x='average_rating',
    y='genres',
    orientation='h',
    text='average_rating',
    hover_data=['rating_count', 'movie_count'],
    title='Average Rating by Genre',
    labels={
        'average_rating': 'Average Rating',
        'genres': 'Genre',
        'rating_count': 'Number of Ratings',
        'movie_count': 'Unique Movies'
    }
)

fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig.update_yaxes(categoryorder='total ascending')
fig.update_xaxes(range=[0, 5])
fig.show()

**✏️ My answer (with numbers):** _(Top and bottom genres, their averages, and how much data each one rests on.)_
- Film-Noir has the highest average rating among the qualifying genres at 3.92, followed by War at 3.82 and Drama at 3.69. Fantasy has the lowest average among qualified genres at 3.22. The unknown genre was excluded because it had only 10 ratings across 2 movies, below the required 100 ratings and 20 unique movies. Overall, viewers appeared most satisfied with Film-Noir and War movies.

## Q3. Ratings over time

**How has the mean rating changed across movie release years?**

Show how many ratings each year is based on, alongside the trend.

In [17]:
# Q3: your code here

# lets create a df ratings_by_year containing an average_rating and average_count column 
ratings_by_year = (
    df.dropna(subset=['year'])
      .groupby('year')
      .agg(
          average_rating=('rating', 'mean'),
          rating_count=('rating', 'count')
      )
      .reset_index()
)

ratings_by_year.head(10)

,year,average_rating,rating_count
0,1922.0,3.555556,54
1,1926.0,3.000000,2
2,1930.0,3.777778,18
3,1931.0,4.000000,44
4,1932.0,3.833333,12
5,1933.0,3.968085,94
6,1934.0,4.024845,161
7,1935.0,3.874016,127
8,1936.0,3.769231,39
9,1937.0,3.745536,224


In [18]:
# lets do some clean up and sort the ratings_by_year df by year in ascending order

ratings_by_year['year'] = ratings_by_year['year'].astype(int) # converting year column to int type for better sorting and display

ratings_by_year['average_rating'] = (
    ratings_by_year['average_rating'].round(2) # round average_rating to two decimal places for better readability
)

ratings_by_year = (
    ratings_by_year
    .sort_values('year')
    .reset_index(drop=True) # drops old index and resets the index to be sequential after sorting
)

ratings_by_year

,year,average_rating,rating_count
0,1922,3.56,54
1,1926,3.00,2
2,1930,3.78,18
3,1931,4.00,44
4,1932,3.83,12
...,...,...,...
66,1994,3.42,9034
67,1995,3.41,12389
68,1996,3.31,18744
69,1997,3.31,15384


In [19]:
# lets plot two charts: line chart for average rating and bar plot rating count per year
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(specs=[[{'secondary_y': True}]])

fig.add_trace(
    go.Bar(
        x=ratings_by_year['year'],
        y=ratings_by_year['rating_count'],
        name='Number of Ratings',
        opacity=0.3
    ),
    secondary_y=True
)

fig.add_trace(
    go.Scatter(
        x=ratings_by_year['year'],
        y=ratings_by_year['average_rating'],
        name='Average Rating',
        mode='lines+markers'
    ),
    secondary_y=False
)

fig.update_layout(
    title='Average Movie Rating by Release Year',
    xaxis_title='Movie Release Year',
    legend_title='Measure',
    hovermode='x unified'
)

fig.update_yaxes(
    title_text='Average Rating',
    range=[0, 5],
    secondary_y=False
)

fig.update_yaxes(
    title_text='Number of Ratings',
    secondary_y=True
)

fig.show()

**✏️ My answer (with numbers):** _(What's the trend? Is it equally believable across the whole timeline?)_
- Mean ratings generally decline for movies with more recent release years. Older decades often have averages near or above 4.0, while movies released from 1995–1998 average about 3.30–3.41. However, the trend is not equally reliable across the timeline. For example, 1942 has the highest average of 4.40 but is based on only 261 ratings from 2 movies, whereas 1996 averages 3.31 based on 18,744 ratings from 297 movies. Therefore, the recent-year averages are supported by much more data, while many early-year averages are more sensitive to a small selection of movies. I excluded 30 ratings from 5 movies with missing release years.

## Q4. Best movies, with a floor

**What are the 5 best-rated movies that have at least 50 ratings? What changes if you raise the floor to 150?**

In [20]:
# Q4: your code here

# calcuate each movies average rating and rating count 
movie_ratings = (
    df.groupby(['movie_id', 'title'])
      .agg(
          average_rating=('rating', 'mean'),
          rating_count=('rating', 'count')
      )
      .reset_index()
)

movie_ratings.head()

,movie_id,title,average_rating,rating_count
0,1,Toy Story (1995),3.878319,452
1,2,GoldenEye (1995),3.206107,131
2,3,Four Rooms (1995),3.033333,90
3,4,Get Shorty (1995),3.550239,209
4,5,Copycat (1995),3.302326,86


In [21]:
# filter by movies with at least 50 ratings, sort by average rating and use rating count as a tie breaker 

top_5_floor_50 = (
    movie_ratings[movie_ratings['rating_count'] >= 50]
    .sort_values(
        ['average_rating', 'rating_count'],
        ascending=[False, False]
    )
    .head(5)
)

top_5_floor_50

,movie_id,title,average_rating,rating_count
407,408,"Close Shave, A (1995)",4.491071,112
317,318,Schindler's List (1993),4.466443,298
168,169,"Wrong Trousers, The (1993)",4.466102,118
482,483,Casablanca (1942),4.456790,243
113,114,Wallace & Gromit: The Best of Aardman Animatio...,4.447761,67


In [22]:
# now lets set floor to 150 ratings
top_5_floor_150 = (
    movie_ratings[movie_ratings['rating_count'] >= 150]
    .sort_values(
        ['average_rating', 'rating_count'],
        ascending=[False, False]
    )
    .head(5)
)

top_5_floor_150

,movie_id,title,average_rating,rating_count
317,318,Schindler's List (1993),4.466443,298
482,483,Casablanca (1942),4.456790,243
63,64,"Shawshank Redemption, The (1994)",4.445230,283
602,603,Rear Window (1954),4.387560,209
11,12,"Usual Suspects, The (1995)",4.385768,267


In [23]:
# now lets combine both top 5 dataframes into one and add a column to indicate the floor value

# .assign() is used to add a new column 
comparison = pd.concat(
    [
        top_5_floor_50.assign(rating_floor='At least 50'),
        top_5_floor_150.assign(rating_floor='At least 150')
    ],
    ignore_index=True
)

comparison['average_rating'] = comparison['average_rating'].round(2)

comparison[
    ['rating_floor', 'title', 'average_rating', 'rating_count']
]

,rating_floor,title,average_rating,rating_count
0,At least 50,"Close Shave, A (1995)",4.49,112
1,At least 50,Schindler's List (1993),4.47,298
2,At least 50,"Wrong Trousers, The (1993)",4.47,118
3,At least 50,Casablanca (1942),4.46,243
4,At least 50,Wallace & Gromit: The Best of Aardman Animatio...,4.45,67
5,At least 150,Schindler's List (1993),4.47,298
6,At least 150,Casablanca (1942),4.46,243
7,At least 150,"Shawshank Redemption, The (1994)",4.45,283
8,At least 150,Rear Window (1954),4.39,209
9,At least 150,"Usual Suspects, The (1995)",4.39,267


In [24]:
# Use the pandas plotting pattern from the Plotly lecture notebook.
pd.options.plotting.backend = 'plotly'

# Put movie titles in the index so they become the horizontal bar labels.
floor_50_ratings = (
    top_5_floor_50
    .set_index('title')['average_rating']
    .sort_values()
)

fig_50 = floor_50_ratings.plot(
    kind='barh',
    title='Top 5 Movies with at Least 50 Ratings',
    labels={'value': 'Average Rating', 'title': 'Movie'}
).update_layout(
    showlegend=False,
    height=500,
    margin=dict(l=320, r=40, t=80, b=60),
    title_x=0.5
).update_xaxes(range=[0, 5], dtick=1)

fig_50.show()

floor_150_ratings = (
    top_5_floor_150
    .set_index('title')['average_rating']
    .sort_values()
)

fig_150 = floor_150_ratings.plot(
    kind='barh',
    title='Top 5 Movies with at Least 150 Ratings',
    labels={'value': 'Average Rating', 'title': 'Movie'}
).update_layout(
    showlegend=False,
    height=500,
    margin=dict(l=320, r=40, t=80, b=60),
    title_x=0.5
).update_xaxes(range=[0, 5], dtick=1)

fig_150.show()

**✏️ My answer (with numbers):** 
- With a minimum of 50 ratings, the top five are *A Close Shave* (4.49, 112 ratings), *Schindler's List* (4.47, 298), *The Wrong Trousers* (4.47, 118), *Casablanca* (4.46, 243), and *Wallace & Gromit: The Best of Aardman Animation* (4.45, 67). At a minimum of 150 ratings, the list becomes *Schindler's List* (4.47, 298), *Casablanca* (4.46, 243), *The Shawshank Redemption* (4.45, 283), *Rear Window* (4.39, 209), and *The Usual Suspects* (4.39, 267). Raising the floor removes three movies supported by only 67–118 ratings and replaces them with movies backed by 209–283 ratings, showing that a higher minimum favors results supported by larger samples even when their averages are slightly lower.

> 🔒 **Stop here.** Parts 1 and 2 should be completely filled in before you open your dashboard.

---
# Part 3: Analysis

Now open your Week 4 dashboard.

## 3.1 How the dashboard got built

Paste 2 prompts that shaped it (from your build log, or from your AI chat history if you didn't keep a log; don't invent any). For each one: what did you ask, what did the AI give you first, and what did you change or accept without checking?

**Prompt 1**
- What I asked: "Let's plan a movie dashboard using Streamlit that addresses the design requirements in this document."
- AI's first attempt: It inspected the CSV and proposed a single-page dashboard with definitions for all four questions, Plotly charts, a genre multiselect, a 50/150 rating-floor control, tests, and deployment steps.
- What I changed (or accepted without checking): I chose the course-first polished style, chart-specific controls, and the modular pandas + Plotly approach. I kept deployment for myself, but I accepted the proposed analytical definitions without independently checking the numbers at that point.

**Prompt 2**
- What I asked: "Execute the plan."
- AI's first attempt: The AI wrote analysis tests first; the first run failed because `analysis.py` did not exist. Those tests defined unique-movie genre counts, rating-weighted genre means, release-year grouping, multi-genre filtering without duplicate ratings, and inclusive rating floors.
- What I changed (or accepted without checking): I kept those analytical choices. I also accepted displaying `unknown` as `Unknown` and using rating count plus title as deterministic tie-breakers.

## 3.2 Compare the answers

Read the answer off your dashboard for each question and put it next to yours.

**Q1**
- Dashboard says: Drama is largest with 725 movies, followed by Comedy with 505; Action and Thriller each have 251, and Romance has 247.
- I say: The same counts, equal to 43.1%, 30.0%, 14.9%, 14.9%, and 14.7% of the 1,682 movies.
- Same? (yes / partly / no): Yes.
- If different, why? There is no numerical difference; both versions deduplicate movies before exploding genres, so each movie counts once in every genre it has.

**Q2**
- Dashboard says: Film-Noir is highest at 3.92, followed by War at 3.82 and Drama at 3.69. `Unknown` is lowest at 3.20, followed by Fantasy at 3.22.
- I say: Film-Noir is highest at 3.92, War is second at 3.82, and Drama is third at 3.69; Fantasy is the lowest qualifying genre at 3.22.
- Same? (yes / partly / no): Partly.
- If different, why? The means match, but the dashboard ranks every genre. My version requires at least 100 ratings and 20 movies, so it excludes `Unknown`, which has only 10 ratings across 2 movies.

**Q3**
- Dashboard says: Mean ratings generally decline for more recent release years; for example, 1942 is 4.40 from 261 ratings, while 1996 is 3.31 from 18,744 ratings. It excludes 30 records with missing release years.
- I say: The same trend and values, while emphasizing that the early-year averages are less reliable because they use much smaller samples.
- Same? (yes / partly / no): Yes.
- If different, why? Both group all rating events by movie release year and drop missing years. I trust the matching values because the dashboard source and my notebook perform the same aggregation.

**Q4**
- Dashboard says: At 50 ratings the top five are *A Close Shave* (4.49, 112), *Schindler's List* (4.47, 298), *The Wrong Trousers* (4.47, 118), *Casablanca* (4.46, 243), and *Wallace & Gromit: The Best of Aardman Animation* (4.45, 67). Raising the floor to 150 replaces the three movies below 150 ratings with *The Shawshank Redemption*, *Rear Window*, and *The Usual Suspects*.
- I say: The same two top-five lists and values.
- Same? (yes / partly / no): Yes.
- If different, why? Both use an inclusive `>=` floor, calculate one mean per movie, and break equal means by rating count.

**When numbers differ, check these first.** Most mismatches come from one of these:

- Counting **ratings** vs. counting **movies**
- One row per rating vs. **one row per rating per genre** (after exploding)
- **Mean of all ratings** vs. the mean of each movie's average
- "At least 50" as ≥ vs. >, or applying the floor before vs. after another step
- What happened to movies with a **missing year**
- Genres with **tiny sample sizes** left in or filtered out

**✏️ Pick two questions and explain what's going on**

**Q2:** The only answer-level mismatch comes from the ranking rule, not from a different mean calculation. The dashboard leaves tiny genres in the ranking, while my minimum of 100 ratings and 20 movies removes `Unknown` and makes Fantasy the lowest qualifying genre.

**Q4:** The lists match because both versions group by movie, keep movies with rating counts greater than or equal to the selected floor, and use rating count as the first tie-breaker. Seeing the same titles, means, and counts at both floors makes this result much more trustworthy than a match based only on rounded averages.

## 3.3 Chart choices

**Q1**
- Chart the dashboard used: A sorted horizontal bar chart with count labels.
- Chart I used: A sorted table of unique-movie genre counts.
- Which communicates the answer better, and why? The dashboard chart communicates the ranking better because bar length makes the large gaps easy to compare and the labels are readable. My written answer adds percentages and the warning that multi-genre percentages overlap.

**Q2**
- Chart the dashboard used: A sorted horizontal bar chart of all genre means, with rating counts available on hover.
- Chart I used: A sorted horizontal bar chart after applying the 100-rating and 20-movie minimums, plus a separate table of excluded genres.
- Which communicates the answer better, and why? My version is better for ranking because it prevents the 10-rating `Unknown` category from being presented as the clear worst genre and documents what was excluded. The dashboard is visually cleaner, but the evidence behind each mean is hidden in hover text.

**Q3**
- Chart the dashboard used: A line chart with markers; rating count appears only in the hover tooltip.
- Chart I used: A line for average rating and bars for rating count on a secondary axis.
- Which communicates the answer better, and why? My chart answers the sample-size part better because the unequal counts are visible without hovering. The dual axes require careful reading, but both axes are labeled and the rating axis honestly runs from 0 to 5.

**Q4**
- Chart the dashboard used: A sorted horizontal bar chart controlled by a 50/150-rating selector, plus a caption describing which titles enter and leave.
- Chart I used: Two separate sorted horizontal bar charts, one for each floor.
- Which communicates the answer better, and why? My two-chart version makes the before-and-after comparison easier because both lists remain visible. The dashboard is more compact and interactive, and its 0–5 axis and full movie labels are honest and readable.

Use what you learned about chart choice and misleading charts (Week 5) and this week's lecture. Is it sorted? Are the labels readable? Are the axes honest? Does it show how much data is behind each number?

## 3.4 What can this data actually tell you?

**Answer at least 3 of these 5**, using your own results as evidence.

**1. Who is in this data?** Check the fewest and the most ratings any single user has given (`df.groupby('user_id').size()`), and look at `rating_year`. What kind of person ends up in a dataset like this, and who is missing? How might that tilt your Q2 answer? *(Week 5: sampling bias)*

**2. How sure are you about the bottom genre?** Take the lowest-rated genre from your Q2. How many ratings *and* how many distinct movies is it built on? Estimate a rough 95% interval: mean ± 1.96 × std / √n. Would you bet it's truly the worst genre? *(Week 5: standard error and confidence intervals. Also ask yourself: are 5,000 ratings of the same handful of movies really 5,000 independent pieces of evidence? Remember the one-row-per-model lesson from last week's homework.)*

**3. What can Q3 really say about time?** How many different values does `rating_year` have? So can this data tell you whether people's taste changed over time? What does your Q3 trend actually describe instead? Why might an old movie in this dataset score higher than a recent one? *(Think: who rates a 1940s film in 1997, and which old movies are even in the data?)*

**4. Does "average" depend on how you average?** Recompute Q2 two ways: (a) average over all ratings, (b) average each movie first, then average the movies within each genre. Does the top genre change? "Which genre do *people* like most?" and "which genre has the best *movies*?" are different questions. *(Week 5: confounding, Simpson's paradox)*

**5. Deployment risk.** A studio exec uses your dashboard to decide which genre to green-light. What's the most misleading conclusion they could draw from one of your charts? Write the one-line caveat you'd put under that chart.

**✏️ Your answers:**

**1. Who is in this data?** Each user supplied at least 20 ratings, and the most active user supplied 737. All 100,000 ratings were submitted in only two years: 52,899 in 1997 and 47,101 in 1998. This is therefore a sample of unusually active late-1990s users, not a representative sample of all movie viewers; casual viewers, nonusers of the service, and present-day audiences are missing, which could tilt genre means toward the tastes of heavy users from that period.

**2. How sure are you about the bottom genre?** Under my qualification rule, Fantasy is lowest at 3.215, based on 1,352 ratings across 22 movies. Its rough 95% interval is `3.215 ± 1.96 × 1.119 / sqrt(1352)`, or about **3.16 to 3.27**. That interval is fairly narrow for rating events, but I still would not confidently call Fantasy inherently worst because 1,352 ratings from only 22 movies are clustered observations rather than 1,352 fully independent pieces of evidence.

**3. What can Q3 really say about time?** `rating_year` has only two values, 1997 and 1998, so this dataset cannot show how the same population's taste changed over many years. Q3 instead compares how users in 1997–1998 rated movies from different release years. Older films may score higher because only selected classics remained visible and attracted ratings, while many ordinary older movies and the people who did not seek them out are absent.

**4. Does average depend on how you average?** Film-Noir remains the top genre in both calculations, but its value changes from **3.92** when every rating is weighted equally to **3.55** when each movie is averaged first and all 24 Film-Noir movies receive equal weight. War is second in both cases, changing from 3.82 to 3.49. The rating-weighted result answers which genres received the highest ratings from people, while the movie-first result answers which genre has the highest average movie, without popular films receiving extra weight.

**5. Deployment risk.** A studio executive could wrongly conclude that making any Film-Noir movie will cause a 3.92 average rating. **Caveat:** These are associations in a selected set of movies rated by active users in 1997–1998; they do not estimate the rating or business performance of a new movie.

## 3.5 Verdict

**Where I'd override the AI** (something it got wrong, hid, or did poorly, and what your version did better): I would override the Q2 ranking because the dashboard treats `Unknown`, supported by only 10 ratings across 2 movies, as the lowest genre. My version applies a stated evidence floor and still shows the excluded category separately. I would also keep my visible rating-count bars in Q3 instead of hiding sample size in hover text.

**Where the AI held up** (something it did well, or a choice you doubted at first that turned out right): The dashboard correctly deduplicates movies before the Q1 genre count, uses release year rather than rating year for Q3, excludes and discloses 30 missing-year records, and applies inclusive rating floors with sensible tie-breakers in Q4. Those choices produced the same values as my independent work.

**One sentence:** In Week 4 I chose the direction and visual style but let the AI drive most analytical and implementation details; in Week 6 I took the wheel by defining the methodology, checking the numbers, and deciding where the dashboard's choices needed limits or clearer evidence.

---
## Optional stretch

Only if you want more practice:

- Pick 4 genres. How does the mean rating change as viewer age increases? (Try more than 4 and see which genres make the most interesting chart.)
- Plot number of ratings vs. mean rating for each movie. Do the most-rated movies get higher ratings? Is there a correlation? *(Week 5: correlation vs. causation)*

---
## Before you submit

- [ ] Copy this notebook into the `exercise/` folder and rename it `Firstname_Lastname_week_06_exercise.ipynb`
- [ ] **Kernel → Restart & Run All** and make sure it runs top to bottom with your charts showing
- [ ] Every ✏️ cell and every fill-in line is filled in
- [ ] Open a PR from your fork, same as every other week